In [ ]:
from copy import copy
import numpy as np
from os import PathLike
import pandas as pd
from pathlib import Path
import re
from typing import Dict

from gtamodel_tools.config import Config

idx = pd.IndexSlice

### Constants

In [ ]:
peak_tps = ['AM', 'PM']
all_tps = ['AM', 'MD', 'PM', 'EV']
all_tps_incl_daily = all_tps + ['daily']

ttc_expr_bus = 'TTC (Express Bus)'
ttc_nonexpr_bus = 'TTC (bus)'
ttc_lrt = 'TTC (LRT)'
ttc_stcar = 'TTC (Streetcar)'
ttc_subway = 'TTC (Subway)'

ttc_operators = [ttc_expr_bus, ttc_nonexpr_bus, ttc_lrt, ttc_stcar, ttc_subway]

# rundir_re = re.compile(r'\d{4}-\d{3}-N\w{1,2}-P\d-E\d-F\d-C\d', re.ASCII)  
rundir_withgrps_re = re.compile(r'(\d{4})-(\d{3})-N(\w{1,2})-P(\d)-E(\d)-F(\d)-C(\d)-{0,1}(\w{0,10})', re.ASCII) 

transit_modes = ['WAT', 'DAT']

tsoi = ['Bloor-Yonge', 'Eglinton-Yonge', 'Pape', 'St. George', 'Union']

### Inputs

In [ ]:
model_root_dir = Path('Complete here')])
summary_support_dir = Path(r'Complete here')
config_fp = summary_support_dir / 'city_of_toronto_2025Network.yml'

output_fp = model_root_dir / 'Run_Summaries.csv'

In [ ]:
# Read the transit peak-hour factors from the config file
config = Config(model_root_dir, config_fp)
t_phfs = {}
for tp, tpdef in config.time_periods.items():
    t_phfs[tp] = tpdef['transit_phf']

### Summary filenames

In [ ]:
line_profiles_fn = '%s_line_profiles.csv'
op_boardings_fn = '%s_boardings_by_operator.csv'
ttc_linked_trips_fn = 'linkedtrips_using_ttc.txt'
rgn2rgn_modeshares_fn = 'daily_region_to_region_trips_by_mode.csv'
tor2cbd_modeshares_fn = 'daily_trips_to_centres_by_mode.csv'
stn_transfers_fn = '%s_stn_transfers.csv'

### Produce the summaries

In [ ]:
def define_scenario(match_obj):
    regroups = match_obj.groups()
    year = regroups[0]
    wfo_scen = regroups[1]
    network = regroups[2]
    pop_scen = regroups[3]
    emp_scen = regroups[4]
    fare_scen = regroups[5]
    congpricing_scen = regroups[6]
    try:
        suffix = regroups[7]
    except Exception as e:
        print(e)
        suffix = None
    
    return (year, pop_scen, emp_scen, wfo_scen, network, fare_scen, congpricing_scen, suffix)

In [ ]:
def process_ttc_boardings_by_mode(fp: PathLike) -> pd.DataFrame:
    s = pd.read_csv(fp, index_col = 'operator')
    ttc_ops_in_file = []
    for to in ttc_operators:
        try:
            tmp = s.loc[to]
            ttc_ops_in_file.append(to)
        except KeyError:
            # print(f'  Operator {to} not found in summary')
            continue
    s2 = s.loc[ttc_ops_in_file].copy()
    if ttc_expr_bus in ttc_ops_in_file:
        colname = s2.columns[0]
        s2.at[ttc_nonexpr_bus, colname] = \
            s2.at[ttc_nonexpr_bus, colname] + s2.at[ttc_expr_bus, colname]
        s2 = s2.drop(ttc_expr_bus)
    return s2

In [ ]:
def set_columns_to_rundetails(df, rs):
    df = df.copy()
    df.columns = pd.MultiIndex.from_arrays(
        [[rs['year']], [rs['pop_scen']], [rs['emp_scen']], [rs['wfo_scen']], [rs['network']], [rs['fare_scen']], [rs['congpricing_scen']], [rs['suffix']]],
        names=['year', 'pop_scen', 'emp_scen', 'wfo_scen', 'network', 'fare_scen', 'congpricing_scen', 'suffix']
    )
    return df

def add_measure_and_time_period_to_index(df, measure, time_period):
    df = df.copy()
    index_name = df.index.name
    df = df.reset_index()
    df['measure'] = measure
    df['time_period'] = time_period
    df = df.set_index(['measure', 'time_period', index_name])
    return df

**Get the run definions from the folder names**

In [ ]:
run_specs = {}
for output_dir in model_root_dir.glob('*'):   
    name = output_dir.stem
    match_obj = rundir_withgrps_re.match(name)
    if match_obj and name != '2041-080-N1-P1-E1-F0-C0':
        year, pop_scen, emp_scen, wfo_scen, network, fare_scen, congpricing_scen, suffix = define_scenario(match_obj)
        run_specs[name] = {
            'year': year,
            'pop_scen': pop_scen,
            'emp_scen': emp_scen,
            'wfo_scen': wfo_scen,
            'network': network,
            'fare_scen': fare_scen,
            'congpricing_scen': congpricing_scen,
            'suffix': suffix
        }

In [ ]:
run_specs

**Peak period load by line (AM and PM periods only)**

In [ ]:
run_list = []
for k, run_spec in run_specs.items():
    tp_list = []
    transit_subdir = model_root_dir / k / 'Summaries' / 'Transit'  
    for tp in peak_tps:
        fp = transit_subdir / (line_profiles_fn % tp)
        df = pd.read_csv(fp, index_col=[0, 1, 2])
        pt = df.groupby(level=0)[['volume']].max()
        pt = set_columns_to_rundetails(pt, run_spec)
        pt = add_measure_and_time_period_to_index(pt, 'pkper_load', tp)
        tp_list.append(pt)
    run_list.append(pd.concat(tp_list, axis=0))
pkper_loads_by_line = pd.concat(run_list, axis=1).sort_index(axis=0).sort_index(axis=1)

# Filter out EELRT lines
to_remove = pkper_loads_by_line.index.get_level_values(2).str.contains('EELRT')
pkper_loads_by_line = pkper_loads_by_line.loc[~to_remove]

**Peak-hour load by line**

In [ ]:
pkhr_loads_by_line = pkper_loads_by_line.copy()
for tp in peak_tps:
    pkhr_loads_by_line.loc[:, idx[tp], :] = (pkhr_loads_by_line.loc[:, idx[tp], :] / t_phfs[tp]).round(0).to_numpy()

# We need to change the measure from peak-period to peak-hour
index_names = pkhr_loads_by_line.index.names
pkhr_loads_by_line = pkhr_loads_by_line.reset_index()
pkhr_loads_by_line[index_names[0]] = 'pkhr_load'
pkhr_loads_by_line = pkhr_loads_by_line.set_index(index_names)

**TTC boardings by mode**

In [ ]:
run_list = []
for k, run_spec in run_specs.items():
    tp_list = []
    transit_subdir = model_root_dir / k / 'Summaries' / 'Transit'  
    for tp in all_tps_incl_daily:
        fp = transit_subdir / (op_boardings_fn % tp)
        df = process_ttc_boardings_by_mode(fp)
        df = set_columns_to_rundetails(df, run_spec)
        df = add_measure_and_time_period_to_index(df, 'oper_boardings', tp)
        tp_list.append(df)
    run_list.append(pd.concat(tp_list, axis=0))
boardings_by_mode = pd.concat(run_list, axis=1).sort_index(axis=0).sort_index(axis=1)

**Linked trips using TTC**

In [ ]:
run_list = []
for k, run_spec in run_specs.items():
    tp_list = []
    transit_subdir = model_root_dir / k / 'Summaries' / 'Transit'  
    fp = transit_subdir / ttc_linked_trips_fn
    with open(fp) as f:
        for line in f:
            line = line.rstrip()
            tp, ltrips = line.split(':')
            df = pd.DataFrame(
                int(float(ltrips)),
                index=pd.Index(['TTC'], name='operator'),
                columns=['temp']
            )
            df = add_measure_and_time_period_to_index(df, 'linked_trips', tp)
            tp_list.append(df)

    run_df = pd.concat(tp_list)   
    run_df = set_columns_to_rundetails(run_df, run_spec)
    run_list.append(run_df)
linked_trips = pd.concat(run_list, axis=1).sort_index(axis=0).sort_index(axis=1)

**Transit mode shares**

In [ ]:
# Toronto-to-Toronto
run_list = []
for k, run_spec in run_specs.items():
    ms_subdir = model_root_dir / k / 'Summaries' / 'Demand'  
    fp = ms_subdir / rgn2rgn_modeshares_fn
    df = pd.read_csv(fp, index_col=[0, 1])
    tor2tor = pd.Series(df.loc[(1,1)])
    tor2tor = pd.DataFrame(
        (tor2tor[transit_modes].sum() / tor2tor.sum()).round(6) * 100.0,
        index = pd.Index(['Toronto-to-Toronto'], name='ODpair'),
        columns=['tor2tor']
    )
    tor2tor = set_columns_to_rundetails(tor2tor, run_spec)
    tor2tor = add_measure_and_time_period_to_index(tor2tor, 'Transit Mode Share', 'daily')
    run_list.append(tor2tor)    
tor2tor_df = pd.concat(run_list, axis=1).sort_index(axis=0).sort_index(axis=1)

In [ ]:
# Toronto-to-Toronto CBD
run_list = []
for k, run_spec in run_specs.items():
    ms_subdir = model_root_dir / k / 'Summaries' / 'Demand'  
    fp = ms_subdir / tor2cbd_modeshares_fn
    df = pd.read_csv(fp, index_col=[0, 1])

    tor_cbd = df.loc[idx[1:16, 'Downtown (TOCore2)'], :]
    sum_tor_cbd = tor_cbd.sum()
    tor2cbd = pd.DataFrame(
        (sum_tor_cbd[transit_modes].sum() / sum_tor_cbd.sum()).round(6) * 100.0,
        index = pd.Index(['Toronto-to-CBD'], name='subgroup'),
        columns=['tor2cbd']
    )
    tor2cbd = set_columns_to_rundetails(tor2cbd, run_spec)
    tor2cbd = add_measure_and_time_period_to_index(tor2cbd, 'Transit Mode Share', 'daily')
    run_list.append(tor2cbd)  
tor2cbd_df = pd.concat(run_list, axis=1).sort_index(axis=0).sort_index(axis=1)

**Station transfers**

In [ ]:
run_list = []
for k, run_spec in run_specs.items():
    tp_list = []
    transit_subdir = model_root_dir / k / 'Summaries' / 'Transit'  
    for tp in all_tps_incl_daily:
        fp = transit_subdir / (stn_transfers_fn % tp)
        df = pd.read_csv(fp, index_col=[0, 1, 2]).squeeze()
        df = df.loc[tsoi].copy()
        df = df.reset_index()
        df['stn_transfer'] = df['stnname'].str.cat([df['from_grp'], df['to_grp']], sep='-')
        df = df.drop(['stnname', 'from_grp', 'to_grp'], axis=1)
        df = df.set_index(['stn_transfer'])
        df = set_columns_to_rundetails(df, run_spec)
        df = add_measure_and_time_period_to_index(df, 'stn_tranfers', tp)      
        tp_list.append(df)
    run_list.append(pd.concat(tp_list, axis=0))

In [ ]:
# Check that they are all unique before trying to concat into a single database
# This will fail if there are NaNs, which will occur if there's a issue with the 
# config that was used in the summary generation
for i in range(0, len(run_list)):
    if not(run_list[i].index.is_unique):
        print(f'Non-unique station transfers:{i}  {run_list[i].columns}')

In [ ]:
stn_transfers_df = pd.concat(run_list, axis=1).sort_index(axis=0).sort_index(axis=1)

# Combine them all together

In [ ]:
combined_results = pd.concat([pkper_loads_by_line, pkhr_loads_by_line, boardings_by_mode, linked_trips, tor2tor_df, tor2cbd_df, stn_transfers_df])

In [ ]:
combined_results

In [ ]:
combined_results.to_csv(output_fp)